# SIFLOW · run_2 · Train MDLM SIFLOW head

Trains the velocity head (frozen MDLM backbone) for 20k steps with SATD + secant + MDM losses. Checkpoints to Drive every 1k steps — if the session times out, just re-run this notebook and it resumes from the latest checkpoint.

**Hardware:** single A100-80GB, <12h. All artifacts are written to Google Drive so the next notebook resumes.

**Needs from previous notebooks:** run_1 (owt_gpt2_256.npy)

In [ ]:
# --- 1. Clone + install (run once per session) ---
REPO_URL = "https://github.com/kagtgi/siflow.git"   # <-- edit to your fork if needed
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# --- 2. Mount Drive + set artifact base (all sessions share this) ---
from siflow.utils import drive
drive.mount()
import os
os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
BASE = drive.base_dir()
print("artifacts ->", BASE)

In [ ]:
!python scripts/train.py --config siflow/config/mdlm.yaml --set \
    data.tokens_path={BASE}/data/owt_gpt2_256.npy \
    out_dir={BASE}/ckpt/mdlm run_id=siflow_mdlm train.total_steps=20000

In [ ]:
# peek at the loss curve
from siflow.analysis.curves import load_jsonl, series
import matplotlib.pyplot as plt
rows = load_jsonl(f"{BASE}/ckpt/mdlm/train_log.jsonl")
for k in ("satd", "vel", "mdm"):
    xs, ys = series(rows, k)
    if xs: plt.plot(xs, ys, label=k)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

Checkpoint is at `{BASE}/ckpt/mdlm/latest.pt`. Proceed to **run_3** for evaluation + figures.